# Recording Videos

In [3]:
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)

In [16]:
import os

from stable_baselines3 import DQN, PPO, A2C

filename = "models/ppo_centipede_model_8000000.zip"

if not os.path.exists(filename):
    raise FileNotFoundError(f"Model file not found: {filename}")

model = PPO.load(
    filename,
    custom_objects={
        "learning_rate": 0.0,
        "lr_schedule": lambda _: 0.0,
        "clip_range": lambda _: 0.0,
    }
)

print("Loaded model:", filename)

Loaded model: models/ppo_centipede_model_8000000.zip


In [17]:
import glob
import time
import base64
from IPython.display import HTML, display

from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, VecVideoRecorder

video_folder = "videos"
os.makedirs(video_folder, exist_ok=True)

eval_env = make_atari_env(
    "ALE/Centipede-v5",
    env_kwargs={"frameskip":1, "repeat_action_probability": 0.0},
    n_envs=1,
    seed=0,
    wrapper_kwargs=dict(terminal_on_life_loss=False),
)
eval_env = VecFrameStack(eval_env, n_stack=4)

video_length = 2000

eval_env = VecVideoRecorder(
    eval_env,
    video_folder=video_folder,
    record_video_trigger=lambda step: step == 0,
    video_length=video_length,
    name_prefix=filename.split(".")[0].split("/")[-1]
)

obs = eval_env.reset()

max_seconds = 60
start_time = time.time()

step_count = 0
done = [False]
episode_reward = 0.0

while (
    step_count < video_length
    and (time.time() - start_time) < max_seconds
    and not done[0]
):
    action, _states = model.predict(obs, deterministic=True)
    obs, rewards, done, infos = eval_env.step(action)
    episode_reward += float(rewards[0])
    step_count += 1

eval_env.close()

elapsed = time.time() - start_time
print(f"Stopped after {step_count} steps and {elapsed:.1f} seconds.")
print(f"Approx reward: {episode_reward:.2f}")

video_files = glob.glob(os.path.join(video_folder, "*.mp4"))
if video_files:
    video_path = sorted(video_files)[-1]
    print("Video saved to:", video_path)

    with open(video_path, "rb") as f:
        mp4 = f.read()
    data_url = "data:video/mp4;base64," + base64.b64encode(mp4).decode()

    display(HTML(f"""
    
        
    
    """))
else:
    print("No video file was created.")

MoviePy - Building video /home/cynthia/csci/reinforcement/videos/ppo_centipede_model_8000000-step-0-to-step-2000.mp4.
MoviePy - Writing video /home/cynthia/csci/reinforcement/videos/ppo_centipede_model_8000000-step-0-to-step-2000.mp4



MoviePy - Done !
MoviePy - video ready /home/cynthia/csci/reinforcement/videos/ppo_centipede_model_8000000-step-0-to-step-2000.mp4
Stopped after 768 steps and 1.6 seconds.
Approx reward: 49.00
Video saved to: videos/ppo_pong_model_8000000-step-0-to-step-2000.mp4
